### In this notebook we show how the model to model evaluation module works

In [1]:
import sys
import os
import json

sys.path.append("../")
sys.path.append("../model_evaluation/")

from model_evaluation.BPMN_conversion import BPMNConverter, XMLBPMNConverter

In [4]:
def load_model(path):
    """Load a BPMN model from a .json (Signavio) or .bpmn/.xml (BPMN 2.0) file.
    Returns the normalised dict ready for the evaluation pipeline.
    """
    if path.endswith(".xml") or path.endswith(".bpmn"):
        return XMLBPMNConverter.convert_file(path).to_dict()
    else:
        with open(path, "r", encoding="utf-8") as fh:
            raw = json.load(fh)
        return BPMNConverter.convert(raw).to_dict()


# Load the first model (JSON or BPMN)
path_model1 = "../examples/misc_credit_quote_creation.json"

# Load the second model (JSON or BPMN)
path_model2 = "../examples/misc_loan_brokerage.json"


model_1_json = load_model(path_model1)
model_2_json = load_model(path_model2)

In [5]:
# ==============================================================================
# BPMN Model Comparison Pipeline
# ==============================================================================
import json

# from rendering import create_similarity_dashboard, print_similarity_report
from bpmn_normalization import normalize_atomic_names

from utils import cosine_sim_optimized
from bpmn_similarity import calculate_bpmn_similarity, calculate_trace_similarity, calculate_hybrid_similarity
from trace_extraction import extract_traces

print("BPMN MODEL COMPARISON PIPELINE")


# Step 1: Model Summary
print("\n[1] MODEL STATISTICS")


def count_elements(model):
    """Count BPMN elements in a model."""
    return {
        "activities": len(model.get("activities", [])),
        "events": len(model.get("events", [])),
        "gateways": len(model.get("gateways", [])),
        "sequence_flows": len(model.get("sequenceFlows", [])),
        "message_flows": len(model.get("messageFlows", [])),
        "pools": len(model.get("pools", [])),
        "lanes": sum(len(p.get("lanes", [])) for p in model.get("pools", [])),
    }


model1_counts = count_elements(model_1_json)
model2_counts = count_elements(model_2_json)

print(f"Model 1: {sum(model1_counts.values())} total elements")
for key, val in model1_counts.items():
    if val > 0:
        print(f"  • {key.replace('_', ' ').title()}: {val}")

print(f"\nModel 2: {sum(model2_counts.values())} total elements")
for key, val in model2_counts.items():
    if val > 0:
        print(f"  • {key.replace('_', ' ').title()}: {val}")

# Step 2: Normalize Names
print("\n[2] SEMANTIC NAME NORMALIZATION")

threshold = 0.6
print(f"Aligning element names using a sentence transformer model (threshold={threshold})...")

model2_aligned, mappings = normalize_atomic_names(model_1_json, model_2_json, cosine_sim_optimized, threshold=threshold)

# from bpmn_sets import extract_bpmn_sets

# print("model1:\n", extract_bpmn_sets(model_1_json), "\nmodel2_aligned:\n", extract_bpmn_sets(model2_aligned))


total_mappings = sum(len(v) for v in mappings.values())


if total_mappings > 0:
    print(f"✓ Applied {total_mappings} semantic name mappings")
    for elem_type, mapping in mappings.items():
        if mapping:
            print(f"  • {elem_type}: {len(mapping)} mappings")
            # Show first example
            first_old, first_new = next(iter(mapping.items()))
            print(f"    Example: '{first_old}' → '{first_new}'")
else:
    print("✓ No mappings needed (names already aligned)")

tr1 = extract_traces(model_1_json, timeout_seconds=5, max_loop_depth=3)

tr2 = extract_traces(model2_aligned, timeout_seconds=5, max_loop_depth=3)

print(f"\nExample traces from model 1 ...\n")
print(tr1.all_traces()[:3])

print(f"\nExample traces from model 2 ...\n")
print(tr2.all_traces()[:3])

# Step 3: Calculate Similarity Without Normalization
print("\n[3] SIMILARITY ANALYSIS")


struct = calculate_bpmn_similarity(model_1_json, model2_aligned, method="jaccard")

beh = calculate_trace_similarity(tr1, tr2, method="jaccard")
hybrid = calculate_hybrid_similarity(struct, beh, structural_weight=0.5)
print(hybrid)

BPMN MODEL COMPARISON PIPELINE

[1] MODEL STATISTICS
Model 1: 30 total elements
  • Activities: 6
  • Events: 2
  • Gateways: 3
  • Sequence Flows: 12
  • Message Flows: 2
  • Pools: 2
  • Lanes: 3

Model 2: 54 total elements
  • Activities: 5
  • Events: 11
  • Gateways: 4
  • Sequence Flows: 18
  • Message Flows: 9
  • Pools: 4
  • Lanes: 3

[2] SEMANTIC NAME NORMALIZATION
Aligning element names using a sentence transformer model (threshold=0.6)...


Trace extraction recovered partial results for net '<unnamed>': 4 sound variant(s), 2 partial trace(s); 2 distinct deadlock marking(s)


✓ Applied 6 semantic name mappings
  • activity_names: 2 mappings
    Example: 'fill out a loan request' → 'Send quote'
  • event_names: 2 mappings
    Example: 'received credit request' → 'Credit request'
  • pool_names: 2 mappings
    Example: 'customer' → 'Customer'

Example traces from model 1 ...

[['Credit request', 'Review  request', 'Calculate  terms', 'Assess risks', 'Prepare contract', 'Send quote', 'Quote sent'], ['Credit request', 'Review  request', 'Assess risks', 'Prepare special terms', 'Prepare contract', 'Send quote', 'Quote sent'], ['Credit request', 'Review  request', 'Calculate  terms', 'Prepare contract', 'Assess risks', 'Send quote', 'Quote sent']]

Example traces from model 2 ...

[['Credit request', 'check credit amount', 'interest rate determination', 'send credit report'], ['Credit request', 'check credit amount', 'send rejection'], ['Quote sent', 'check credit amount', 'send rejection']]

[3] SIMILARITY ANALYSIS
{'structural': 0.08619769119769119, 'behavioral

In [6]:
from rendering.dashboard import BPMNSimilarityDashboard
from bpmn_normalization import normalize_atomic_names
from bpmn_similarity import (
    calculate_bpmn_similarity,
    calculate_trace_similarity,
    calculate_hybrid_similarity,
)
from trace_extraction import extract_traces
from utils.string_similarity import cosine_sim_optimized

# from edge_case_fixtures import (
#     FIXTURE_1_A,
#     FIXTURE_1_B,
#     FIXTURE_2_A,
#     FIXTURE_2_B,
#     FIXTURE_3_A,
#     FIXTURE_3_B,
#     FIXTURE_4_A,
#     FIXTURE_4_B,
#     FIXTURE_5_A,
#     FIXTURE_5_B,
#     FIXTURE_6_A,
#     FIXTURE_6_B,
# )

# ... usual dashboard imports ...

dashboard = BPMNSimilarityDashboard(
    model_1_json,
    model_2_json,
    similarity_func=cosine_sim_optimized,
    calculate_similarity_func=calculate_bpmn_similarity,
    normalize_func=normalize_atomic_names,
    extract_traces_func=extract_traces,
    calculate_trace_similarity_func=calculate_trace_similarity,
    calculate_hybrid_func=calculate_hybrid_similarity,
    initial_threshold=0.8,
)
dashboard.display()